# FOG: minimal audited GPU trial
Attach the private trial bundle. Select a free GPU and enable Internet. Run all cells. This uses the current checkpoint and a tiny recovered corpus solely to verify the training pipeline; it does not establish improved playing strength. Both five-game audits remain enabled.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

WORK = Path("/kaggle/working/fog")
INPUT = Path("/kaggle/input")
if not (WORK / "bundle_manifest.json").is_file():
    manifests = list(INPUT.rglob("bundle_manifest.json"))
    archives = list(INPUT.rglob("fog-kaggle-trial.zip"))
    if len(manifests) == 1:
        if WORK.exists():
            raise RuntimeError("The work directory already exists without a bundle manifest; inspect it first.")
        shutil.copytree(manifests[0].parent, WORK)
    elif not manifests and len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            for name in archive.namelist():
                target = (WORK.parent / name).resolve()
                if not target.is_relative_to(WORK.resolve()):
                    raise ValueError(f"Unexpected archive path: {name}")
            archive.extractall(WORK.parent)
    else:
        raise RuntimeError("Attach exactly one FOG bundle dataset, then rerun this cell.")
os.chdir(WORK)
print(json.loads((WORK / "bundle_manifest.json").read_text()))


In [ ]:
# Installation needs Internet. The dataset already contains the built simulator.
if not shutil.which("node") or not shutil.which("npm"):
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "nodejs", "npm"], check=True)
subprocess.run([sys.executable, "-m", "fog.scripts.kaggle_setup"], cwd=WORK, check=True)
PYTHON = str(WORK / ".venv/bin/python")


In [ ]:
# This runs real PPO updates, then checks changed weights and saved optimizer state.
# Export logs even when setup audits or training fail.
try:
    subprocess.run([PYTHON, "-m", "fog.scripts.kaggle_trial", "--device", "cuda"], check=True)
finally:
    output = WORK / "runs/gen9ou_kaggle_trial"
    if output.exists():
        shutil.copytree(WORK / "runs/kaggle_environment", output / "environment", dirs_exist_ok=True)
        shutil.make_archive("/kaggle/working/fog-trial-output", "zip", output)
report = json.loads((output / "trial_report.json").read_text())
print(json.dumps(report, indent=2))
assert report["passed"]
